# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Mã nguồn được module hoá trong thư mục `src/`.

## 0. Môi trường (Colab T4 / Server)

In [ ]:
!nvidia-smi
# !pip install -r requirements-colab.txt

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    pass

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from pathlib import Path
from scripts.labcommon import read_jsonl

meta = json.loads(Path("data/splits/meta.json").read_text())
print("dataset_version:", meta.get("dataset_version"))
for s, info in meta.get("counts", {}).items():
    print(f"{s}: {info['images']} ảnh, boxes: {info['boxes']}")

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from src.label_utils import yolo_to_box, objects_to_answer, box_to_pixel, PROMPT_NORM1000
train_samples = read_jsonl("data/splits/train.jsonl")[:2]
for sample in train_samples:
    print(sample["image_id"], "->", objects_to_answer(sample["objects"]))

## 3. E0: zero-shot

In [ ]:
!python src/infer.py --config configs/e0.yaml --name E0 --split test

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from src.dataset import RoadDamageDataset, encode_example
ds = RoadDamageDataset("data/splits/train.jsonl", data_root="data")
print("Train samples:", len(ds))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from src.model_utils import apply_setup
print("Setup module loaded successfully!")

## 6. Vòng huấn luyện

In [ ]:
# Huấn luyện E1
!python src/train.py --config configs/e1.yaml
# Huấn luyện E2
!python src/train.py --config configs/e2.yaml
# Huấn luyện E3
!python src/train.py --config configs/e3.yaml

## 7. Suy luận và chấm E1–E3

In [ ]:
!python src/infer.py --config configs/e1.yaml --name E1 --split test
!python src/infer.py --config configs/e2.yaml --name E2 --split test
!python src/infer.py --config configs/e3.yaml --name E3 --split test

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: Ablation rank 4 vs rank 64
!python src/train.py --config configs/e1.yaml --name E4_rank4 --rank 4 --output-dir work/checkpoints/e4_rank4
!python src/infer.py --config configs/e1.yaml --name E4_rank4 --checkpoint-dir work/checkpoints/e4_rank4 --evidence-out report/evidence/e4_rank_4.json --split test

!python src/train.py --config configs/e1.yaml --name E4_rank64 --rank 64 --output-dir work/checkpoints/e4_rank64
!python src/infer.py --config configs/e1.yaml --name E4_rank64 --checkpoint-dir work/checkpoints/e4_rank64 --evidence-out report/evidence/e4_rank_64.json --split test